## 1. Khai báo thư viện và thiết lập đường dẫn

Notebook này thực hiện bước khai thác tập phổ biến từ ma trận giao dịch - sản phẩm đã được chuẩn bị ở giai đoạn trước. Các thuật toán chính được sử dụng là FP-Growth và Apriori.

Trong đó, FP-Growth được dùng làm thuật toán chính vì phù hợp hơn với dữ liệu giao dịch có số lượng sản phẩm lớn và ma trận thưa. Apriori được dùng như thuật toán baseline để so sánh thời gian xử lý và số lượng tập phổ biến sinh ra.

Kết quả của notebook này sẽ được lưu vào thư mục `results`, bao gồm frequent itemsets và bảng tổng hợp thực nghiệm.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import time
import warnings

warnings.filterwarnings("ignore")

cwd = Path.cwd()

if cwd.name.lower() == "notebooks":
    PROJECT_DIR = cwd.parent
else:
    PROJECT_DIR = cwd

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

PROJECT_DIR: d:\HK2_NAM3\KTDL_KPTT\CUOI_KY
PROCESSED_DIR: d:\HK2_NAM3\KTDL_KPTT\CUOI_KY\data\processed
RESULTS_DIR: d:\HK2_NAM3\KTDL_KPTT\CUOI_KY\results


## 2. Kiểm tra thư viện mlxtend

Thư viện `mlxtend` cung cấp sẵn các hàm triển khai Apriori, FP-Growth và Association Rules. Trong notebook này, ta cần sử dụng `fpgrowth` và `apriori` để khai thác frequent itemsets.

Nếu môi trường Python chưa có `mlxtend`, cần cài đặt trước khi tiếp tục chạy các cell xử lý thuật toán.

In [4]:
try:
    from mlxtend.frequent_patterns import fpgrowth, apriori
    print("Trạng thái: Thư viện mlxtend đã sẵn sàng.")
except ImportError:
    print("Trạng thái: Chưa có thư viện mlxtend.")
    print("Hãy chạy lệnh sau trong notebook hoặc terminal:")
    print("pip install mlxtend")

Trạng thái: Thư viện mlxtend đã sẵn sàng.


## 3. Đọc dữ liệu đầu vào cho khai thác tập phổ biến

Bước này đọc hai file quan trọng từ thư mục `data/processed`.

`basket_matrix_sparse.pkl` là ma trận giao dịch - sản phẩm. Mỗi dòng tương ứng với một đơn hàng, mỗi cột tương ứng với một sản phẩm, và giá trị trong ma trận cho biết sản phẩm có xuất hiện trong đơn hàng hay không.

`product_mapping_sample.csv` dùng để ánh xạ `product_id` sang tên sản phẩm, quầy hàng và nhóm ngành hàng. File này giúp kết quả frequent itemsets dễ đọc hơn thay vì chỉ hiển thị mã sản phẩm.

In [5]:
basket_matrix = pd.read_pickle(PROCESSED_DIR / "basket_matrix_sparse.pkl")
product_mapping = pd.read_csv(PROCESSED_DIR / "product_mapping_sample.csv")

print("Trạng thái: Load dữ liệu thành công.")
print("Kích thước basket_matrix:", basket_matrix.shape)
print("Kích thước product_mapping:", product_mapping.shape)

display(product_mapping.head())

Trạng thái: Load dữ liệu thành công.
Kích thước basket_matrix: (75519, 3000)
Kích thước product_mapping: (3000, 4)


,product_id,product_name,aisle_name,department_name
0,35108,Salted Butter,butter,dairy eggs
1,40593,Cream Cheese,other creams cheeses,dairy eggs
2,17461,Air Chilled Organic Boneless Skinless Chicken ...,poultry counter,meat seafood
3,22825,Organic D'Anjou Pears,fresh fruits,produce
4,25256,Cultured Low Fat Buttermilk,milk,dairy eggs


## 4. Chuyển ma trận giao dịch sang dạng boolean

Các thuật toán Apriori và FP-Growth cần dữ liệu đầu vào ở dạng nhị phân hoặc boolean. Vì `basket_matrix` đang biểu diễn sự xuất hiện của sản phẩm bằng giá trị `0` và `1`, ta chuyển ma trận này sang dạng `True` và `False`.

Trong đó:

| Giá trị | Ý nghĩa |
| --- | --- |
| `True` | Sản phẩm xuất hiện trong đơn hàng |
| `False` | Sản phẩm không xuất hiện trong đơn hàng |

Việc chuyển sang boolean giúp dữ liệu phù hợp hơn với yêu cầu đầu vào của thư viện `mlxtend`.

In [6]:
basket_bool = basket_matrix.astype(bool)

print("Trạng thái: Đã chuyển basket_matrix sang dạng boolean.")
print("Kích thước basket_bool:", basket_bool.shape)

display(basket_bool.head())

Trạng thái: Đã chuyển basket_matrix sang dạng boolean.
Kích thước basket_bool: (75519, 3000)


product_id,10,34,45,49,79,95,116,117,130,141,...,49519,49520,49533,49583,49585,49605,49610,49621,49628,49683
order_id,,,,,,,,,,,,,,,,,,,,,
28,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
67,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
71,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
109,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
122,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## 5. Tạo hàm ánh xạ itemset sang tên sản phẩm

Kết quả của thuật toán thường trả về frequent itemsets dưới dạng tập hợp các `product_id`. Tuy nhiên, các mã sản phẩm này khó đọc khi trình bày trong báo cáo.

Vì vậy, bước này tạo một từ điển ánh xạ từ `product_id` sang `product_name`, sau đó xây dựng hàm chuyển một itemset từ dạng mã sản phẩm sang dạng tên sản phẩm.

Ví dụ:

`{24852, 21137}`

có thể được hiển thị thành:

`Banana, Organic Strawberries`

In [8]:
product_mapping["product_id"] = product_mapping["product_id"].astype(str)

product_id_to_name = dict(
    zip(
        product_mapping["product_id"],
        product_mapping["product_name"]
    )
)

def itemset_to_names(itemset):
    names = []
    for item in itemset:
        item_str = str(item)
        product_name = product_id_to_name.get(item_str, item_str)
        names.append(product_name)
    return ", ".join(sorted(names))

print("Trạng thái: Đã tạo ánh xạ product_id sang product_name.")
print("Số sản phẩm trong mapping:", len(product_id_to_name))

Trạng thái: Đã tạo ánh xạ product_id sang product_name.
Số sản phẩm trong mapping: 3000


## 6. Chạy thử FP-Growth với ngưỡng support cao

Trước khi chạy nhiều ngưỡng khác nhau, notebook chạy thử FP-Growth với `min_support = 0.01`. Đây là ngưỡng tương đối cao, giúp thuật toán chạy nhanh và kiểm tra xem dữ liệu đầu vào có phù hợp hay không.

Với 75,519 giao dịch, `min_support = 0.01` có nghĩa là một sản phẩm hoặc nhóm sản phẩm phải xuất hiện trong khoảng 755 đơn hàng trở lên mới được xem là phổ biến.

Nếu bước chạy thử này thành công, có thể tiếp tục chạy FP-Growth với nhiều ngưỡng support thấp hơn để tạo nhiều frequent itemsets hơn.

In [9]:
min_support_test = 0.01

start_time = time.time()

frequent_itemsets_test = fpgrowth(
    basket_bool,
    min_support=min_support_test,
    use_colnames=True
)

runtime = time.time() - start_time

frequent_itemsets_test["length"] = frequent_itemsets_test["itemsets"].apply(len)
frequent_itemsets_test["itemset_names"] = frequent_itemsets_test["itemsets"].apply(itemset_to_names)

frequent_itemsets_test = frequent_itemsets_test[
    ["itemsets", "itemset_names", "support", "length"]
].sort_values("support", ascending=False)

print("Thuật toán: FP-Growth")
print("min_support:", min_support_test)
print("Số frequent itemsets:", len(frequent_itemsets_test))
print("Thời gian chạy:", round(runtime, 2), "giây")

display(frequent_itemsets_test.head(20))

Thuật toán: FP-Growth
min_support: 0.01
Số frequent itemsets: 124
Thời gian chạy: 5.53 giây


,itemsets,itemset_names,support,length
11,frozenset({24852}),Banana,0.143884,1
3,frozenset({13176}),Bag of Organic Bananas,0.121837,1
13,frozenset({21137}),Organic Strawberries,0.087024,1
16,frozenset({21903}),Organic Baby Spinach,0.077874,1
0,frozenset({47209}),Organic Hass Avocado,0.069784,1
44,frozenset({47766}),Organic Avocado,0.059561,1
4,frozenset({26209}),Limes,0.049166,1
1,frozenset({47626}),Large Lemon,0.047789,1
42,frozenset({27845}),Organic Whole Milk,0.046465,1
19,frozenset({16797}),Strawberries,0.046372,1


## 7. Chạy FP-Growth với nhiều ngưỡng support

Sau khi bước chạy thử thành công, notebook tiếp tục chạy FP-Growth với nhiều giá trị `min_support` khác nhau. Việc thử nhiều ngưỡng giúp quan sát mối quan hệ giữa ngưỡng support và số lượng frequent itemsets.

Các ngưỡng được sử dụng gồm:

| min_support | Ý nghĩa gần đúng |
| --- | --- |
| `0.01` | Xuất hiện trong khoảng 755 giao dịch trở lên |
| `0.005` | Xuất hiện trong khoảng 378 giao dịch trở lên |
| `0.002` | Xuất hiện trong khoảng 151 giao dịch trở lên |
| `0.001` | Xuất hiện trong khoảng 76 giao dịch trở lên |

Khi `min_support` giảm, số lượng frequent itemsets thường tăng lên. Tuy nhiên, nếu ngưỡng quá thấp, số lượng tập phổ biến có thể tăng mạnh và gây khó khăn cho việc phân tích.

In [10]:
fpgrowth_support_values = [0.01, 0.005, 0.002, 0.001]

fpgrowth_results = {}
experiment_rows = []

for min_support in fpgrowth_support_values:
    print("Đang chạy FP-Growth với min_support =", min_support)
    
    start_time = time.time()
    
    itemsets = fpgrowth(
        basket_bool,
        min_support=min_support,
        use_colnames=True
    )
    
    runtime = time.time() - start_time
    
    itemsets["length"] = itemsets["itemsets"].apply(len)
    itemsets["itemset_names"] = itemsets["itemsets"].apply(itemset_to_names)
    
    itemsets = itemsets[
        ["itemsets", "itemset_names", "support", "length"]
    ].sort_values("support", ascending=False)
    
    fpgrowth_results[min_support] = itemsets
    
    experiment_rows.append({
        "algorithm": "FP-Growth",
        "min_support": min_support,
        "num_frequent_itemsets": len(itemsets),
        "max_itemset_length": itemsets["length"].max() if len(itemsets) > 0 else 0,
        "runtime_seconds": round(runtime, 2)
    })
    
    print("Hoàn thành. Số itemsets:", len(itemsets), "| Thời gian:", round(runtime, 2), "giây")
    print("-" * 80)

fpgrowth_summary = pd.DataFrame(experiment_rows)
display(fpgrowth_summary)

Đang chạy FP-Growth với min_support = 0.01
Hoàn thành. Số itemsets: 124 | Thời gian: 5.51 giây
--------------------------------------------------------------------------------
Đang chạy FP-Growth với min_support = 0.005
Hoàn thành. Số itemsets: 357 | Thời gian: 5.81 giây
--------------------------------------------------------------------------------
Đang chạy FP-Growth với min_support = 0.002
Hoàn thành. Số itemsets: 1510 | Thời gian: 7.03 giây
--------------------------------------------------------------------------------
Đang chạy FP-Growth với min_support = 0.001
Hoàn thành. Số itemsets: 4269 | Thời gian: 8.04 giây
--------------------------------------------------------------------------------


,algorithm,min_support,num_frequent_itemsets,max_itemset_length,runtime_seconds
0,FP-Growth,0.010,124,2,5.51
1,FP-Growth,0.005,357,2,5.81
2,FP-Growth,0.002,1510,3,7.03
3,FP-Growth,0.001,4269,4,8.04


## 8. Chọn ngưỡng support chính cho bước sinh luật

Sau khi chạy FP-Growth với nhiều ngưỡng support, cần chọn một ngưỡng chính để sử dụng cho bước sinh luật kết hợp.

Trong bài này, `min_support = 0.001` được chọn làm ngưỡng chính vì tương ứng với khoảng 76 giao dịch trên tổng 75,519 giao dịch. Ngưỡng này đủ thấp để phát hiện nhiều mẫu mua kèm hơn, nhưng vẫn đủ cao để loại bỏ các tập sản phẩm xuất hiện quá ít.

Kết quả tại ngưỡng này sẽ được dùng làm đầu vào cho bước sinh association rules.

In [11]:
selected_min_support = 0.001

frequent_itemsets_selected = fpgrowth_results[selected_min_support].copy()

print("Ngưỡng support được chọn:", selected_min_support)
print("Số frequent itemsets:", len(frequent_itemsets_selected))
print("Độ dài itemset lớn nhất:", frequent_itemsets_selected["length"].max())

display(frequent_itemsets_selected.head(20))

Ngưỡng support được chọn: 0.001
Số frequent itemsets: 4269
Độ dài itemset lớn nhất: 4


,itemsets,itemset_names,support,length
56,frozenset({24852}),Banana,0.143884,1
16,frozenset({13176}),Bag of Organic Bananas,0.121837,1
62,frozenset({21137}),Organic Strawberries,0.087024,1
74,frozenset({21903}),Organic Baby Spinach,0.077874,1
0,frozenset({47209}),Organic Hass Avocado,0.069784,1
205,frozenset({47766}),Organic Avocado,0.059561,1
23,frozenset({26209}),Limes,0.049166,1
1,frozenset({47626}),Large Lemon,0.047789,1
199,frozenset({27845}),Organic Whole Milk,0.046465,1
89,frozenset({16797}),Strawberries,0.046372,1


## 9. Chạy Apriori làm thuật toán baseline

Apriori được sử dụng như thuật toán baseline để so sánh với FP-Growth. Cả hai thuật toán đều nhằm mục tiêu tìm frequent itemsets, nhưng cách hoạt động khác nhau.

Apriori sinh các tập ứng viên theo từng cấp độ rồi đếm support để lọc. Cách tiếp cận này dễ hiểu nhưng có thể chậm khi số lượng sản phẩm lớn. Trong khi đó, FP-Growth nén dữ liệu vào cấu trúc FP-tree và khai thác mẫu phổ biến mà không cần sinh ứng viên theo cách truyền thống.

Do Apriori có thể chạy lâu hơn, notebook chỉ thử các ngưỡng `0.01`, `0.005` và `0.002`.

In [12]:
apriori_support_values = [0.01, 0.005, 0.002,0.001]

apriori_results = {}
apriori_experiment_rows = []

for min_support in apriori_support_values:
    print("Đang chạy Apriori với min_support =", min_support)
    
    start_time = time.time()
    
    itemsets = apriori(
        basket_bool,
        min_support=min_support,
        use_colnames=True,
        low_memory=True
    )
    
    runtime = time.time() - start_time
    
    itemsets["length"] = itemsets["itemsets"].apply(len)
    itemsets["itemset_names"] = itemsets["itemsets"].apply(itemset_to_names)
    
    itemsets = itemsets[
        ["itemsets", "itemset_names", "support", "length"]
    ].sort_values("support", ascending=False)
    
    apriori_results[min_support] = itemsets
    
    apriori_experiment_rows.append({
        "algorithm": "Apriori",
        "min_support": min_support,
        "num_frequent_itemsets": len(itemsets),
        "max_itemset_length": itemsets["length"].max() if len(itemsets) > 0 else 0,
        "runtime_seconds": round(runtime, 2)
    })
    
    print("Hoàn thành. Số itemsets:", len(itemsets), "| Thời gian:", round(runtime, 2), "giây")
    print("-" * 80)

apriori_summary = pd.DataFrame(apriori_experiment_rows)
display(apriori_summary)

Đang chạy Apriori với min_support = 0.01
Hoàn thành. Số itemsets: 124 | Thời gian: 11.1 giây
--------------------------------------------------------------------------------
Đang chạy Apriori với min_support = 0.005
Hoàn thành. Số itemsets: 357 | Thời gian: 18.41 giây
--------------------------------------------------------------------------------
Đang chạy Apriori với min_support = 0.002
Hoàn thành. Số itemsets: 1510 | Thời gian: 39.34 giây
--------------------------------------------------------------------------------
Đang chạy Apriori với min_support = 0.001
Hoàn thành. Số itemsets: 4269 | Thời gian: 66.02 giây
--------------------------------------------------------------------------------


,algorithm,min_support,num_frequent_itemsets,max_itemset_length,runtime_seconds
0,Apriori,0.010,124,2,11.10
1,Apriori,0.005,357,2,18.41
2,Apriori,0.002,1510,3,39.34
3,Apriori,0.001,4269,4,66.02


## 10. Tổng hợp kết quả thực nghiệm FP-Growth và Apriori

Sau khi chạy cả FP-Growth và Apriori, notebook tổng hợp kết quả vào một bảng chung để so sánh.

Các tiêu chí so sánh gồm:

| Tiêu chí | Ý nghĩa |
| --- | --- |
| `algorithm` | Thuật toán được sử dụng |
| `min_support` | Ngưỡng support |
| `num_frequent_itemsets` | Số lượng frequent itemsets sinh ra |
| `max_itemset_length` | Độ dài lớn nhất của itemset |
| `runtime_seconds` | Thời gian chạy tính bằng giây |

Nếu cùng một ngưỡng support, FP-Growth và Apriori cho số lượng frequent itemsets giống nhau nhưng FP-Growth chạy nhanh hơn, đây là bằng chứng thực nghiệm cho thấy FP-Growth phù hợp hơn với bộ dữ liệu này.

In [13]:
experiment_summary = pd.concat(
    [fpgrowth_summary, apriori_summary],
    ignore_index=True
).sort_values(["min_support", "algorithm"])

display(experiment_summary)

,algorithm,min_support,num_frequent_itemsets,max_itemset_length,runtime_seconds
7,Apriori,0.001,4269,4,66.02
3,FP-Growth,0.001,4269,4,8.04
6,Apriori,0.002,1510,3,39.34
2,FP-Growth,0.002,1510,3,7.03
5,Apriori,0.005,357,2,18.41
1,FP-Growth,0.005,357,2,5.81
4,Apriori,0.010,124,2,11.10
0,FP-Growth,0.010,124,2,5.51


Kết quả thực nghiệm cho thấy khi min_support giảm từ 0.01 xuống 0.001, số lượng frequent itemsets tăng từ 124 lên 4,269. Điều này phù hợp với lý thuyết vì ngưỡng support càng thấp thì càng có nhiều tập sản phẩm thỏa điều kiện phổ biến.

Ở cùng các ngưỡng min_support, Apriori và FP-Growth tạo ra số lượng frequent itemsets giống nhau. Tuy nhiên, FP-Growth có thời gian xử lý thấp hơn đáng kể. Cụ thể, tại min_support = 0.002, FP-Growth xử lý trong khoảng 7.78 giây, trong khi Apriori cần khoảng 42.64 giây. Điều này cho thấy FP-Growth phù hợp hơn với bộ dữ liệu giao dịch có quy mô lớn và ma trận thưa.

Vì vậy, nhóm sử dụng Apriori như thuật toán baseline để so sánh, còn FP-Growth được chọn làm thuật toán chính cho bước khai thác tập phổ biến. Ngưỡng min_support = 0.001 được chọn cho bước tiếp theo vì tạo ra 4,269 frequent itemsets, đủ phong phú để sinh luật kết hợp, đồng thời thời gian xử lý vẫn ở mức chấp nhận được.

## 11. Lưu frequent itemsets và kết quả thực nghiệm

Kết quả frequent itemsets được lưu thành hai định dạng:

| File | Ý nghĩa |
| --- | --- |
| `frequent_itemsets.csv` | Dùng để xem kết quả bằng Excel hoặc đưa vào báo cáo |
| `frequent_itemsets.pkl` | Dùng cho bước sinh association rules vì giữ nguyên kiểu dữ liệu `frozenset` của cột `itemsets` |
| `experiment_frequent_itemsets_summary.csv` | Lưu bảng so sánh thực nghiệm giữa FP-Growth và Apriori |

File `.pkl` rất quan trọng cho bước tiếp theo, vì nếu chỉ lưu `itemsets` ở dạng CSV, tập sản phẩm sẽ bị chuyển thành chuỗi và khó dùng trực tiếp cho hàm sinh luật kết hợp.

In [14]:
experiment_summary.to_csv(
    RESULTS_DIR / "experiment_frequent_itemsets_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

frequent_itemsets_selected.to_csv(
    RESULTS_DIR / "frequent_itemsets.csv",
    index=False,
    encoding="utf-8-sig"
)

frequent_itemsets_selected.to_pickle(
    RESULTS_DIR / "frequent_itemsets.pkl"
)

print("Đã xuất file experiment_frequent_itemsets_summary.csv")
print("Đã xuất file frequent_itemsets.csv")
print("Đã xuất file frequent_itemsets.pkl")

Đã xuất file experiment_frequent_itemsets_summary.csv
Đã xuất file frequent_itemsets.csv
Đã xuất file frequent_itemsets.pkl
